# Practice 1

 # 📓 Exercise 1 — Complete Standalone Notebook

# Exercise 1: Model Comparison with 5-Fold Cross-Validation

**Goal:** Extend Section 8's model comparison by adding a third candidate regressor (`GradientBoostingRegressor`) alongside Linear Regression and Random Forest, evaluate all three using 5-fold Cross-Validation on the training set, and rank them by their CV Root Mean Squared Error (RMSE).

# Cell 1 — Import the required libraries

In this cell, we import all necessary libraries and modules for data manipulation, preprocessing, pipeline building, regression modeling, and cross-validation evaluation.

In [13]:
# Import core tools, preprocessing modules, and regression models.

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Import candidate regressor models

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Cell 2 — Load and prepare the California Housing dataset

We load the dataset, introduce missing values and engineered features as specified in Notebook 2, and prepare the target variable.

In [14]:
# Load dataset and create reproducible missing values and engineered features.

california = fetch_california_housing(as_frame=True)
housing = california.frame

rng = np.random.default_rng(42)
messy = housing.copy()

# Add missing values for AveBedrms and AveRooms

missing_idx = rng.choice(messy.index, size=200, replace=False)
messy.loc[missing_idx, "AveBedrms"] = np.nan
missing_idx_2 = rng.choice(messy.index, size=120, replace=False)
messy.loc[missing_idx_2, "AveRooms"] = np.nan

# Feature engineering: Categorical bins and numerical ratios

messy["coast_region"] = pd.cut(
    messy["Longitude"],
    bins=[-125, -121.22, -118.20, -114],
    labels=["Coastal", "Central", "Inland"],)

messy["income_cat"] = pd.cut(
    messy["MedInc"],
    bins=[0, 2.5, 4.5, 7, np.inf],
    labels=["Low", "Medium", "High", "Very High"],)

messy["bedrooms_ratio"] = messy["AveBedrms"] / messy["AveRooms"]
messy["rooms_per_person"] = messy["AveRooms"] / messy["AveOccup"]

X = messy.drop(columns="MedHouseVal")
y = messy["MedHouseVal"]

# Cell 3 — Split data into training and test sets

We split the features and target into training (80%) and testing (20%) sets using a fixed random state for full reproducibility.

In [15]:
# Perform train-test split (80% train, 20% test).

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Define feature groups for preprocessing

numeric_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"]

categorical_features = ["coast_region"]

# Cell 4 — Build the preprocessing pipeline

We construct a ColumnTransformer to handle missing value imputation, standard scaling for numeric features, and one-hot encoding for categorical features.

In [16]:
# Define numerical and categorical preprocessing sub-pipelines.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),])


# Combine into a single ColumnTransformer preprocessor.

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),])

# Cell 5 — Define candidate models

We set up a dictionary containing three regression candidates: Linear Regression, Random Forest, and Gradient Boosting Regressor.

In [17]:
# Define candidate regressor models.

candidates = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1),
        
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, random_state=42),}

# Cell 6 — Perform 5-fold Cross-Validation

We loop over each candidate model, attach it to the preprocessor pipeline, and calculate the mean 5-fold cross-validation RMSE using training data only.

In [18]:
# Evaluate candidates using 5-fold cross-validation RMSE.
cv_results = {}

# Build complete pipeline combining preprocessing and candidate model
for name, model in candidates.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    
    # Calculate 5-fold CV scores

    fold_scores = cross_val_score(
        pipe, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
    )
    
    # Convert negative RMSE to positive RMSE and average over 5 folds
    
    cv_results[name] = -fold_scores.mean()

cv_results

{'Linear Regression': np.float64(0.718425834212262),
 'Random Forest': np.float64(0.5090526847162172),
 'Gradient Boosting': np.float64(0.5351619281964658)}

# Cell 7 — Build and rank the comparison table

We assemble the CV RMSE scores into a formatted pandas DataFrame and rank the candidates from best (lowest RMSE) to worst.

In [ ]:
# Create a pandas DataFrame and rank models by CV RMSE.

cv_table = pd.DataFrame(
    list(cv_results.items()), columns=["Model", "CV RMSE"])

# Sort models by CV RMSE in ascending order (lower error is better)

cv_table = cv_table.sort_values(by="CV RMSE", ascending=True).reset_index(drop=True)
cv_table.index += 1
cv_table.index.name = "Rank"

cv_table.round(4)

,Model,CV RMSE
Rank,,
1,Random Forest,0.5091
2,Gradient Boosting,0.5352
3,Linear Regression,0.7184


# Cell 8 — Final interpretation

In this cell, we extract the top-performing model, display the final rankings, and summarize why tree-based ensemble methods outperform linear regression on this tabular dataset.

In [20]:
# Report the best candidate model based on Cross-Validation score.

best_model = cv_table.iloc[0]

print(f"Top-Ranked Model: {best_model['Model']}")
print(f"Lowest CV RMSE:   {best_model['CV RMSE']:.4f}")

Top-Ranked Model: Random Forest
Lowest CV RMSE:   0.5091
